# CMV LLM annotation: batch runner

Runs the annotation prompts on **GPT-6.1 Sol** (OpenAI) and **Claude Sonnet 5.5** (Anthropic) for the 1,725 main-round pairs: 3 metrics × 2 A/B orders = **10,350 requests per model**.

Run the cells top to bottom. Every cell is safe to re-run: submitted batch IDs are saved to `runs/state.json`, so if the session disconnects, reopen the notebook, run sections 1–2, then continue from section 6.

**Reproducing the thesis does not require running this notebook.** The batch request files can be rebuilt without any API call (section 3; they come out byte-identical to the ones submitted), and the raw API responses and parsed annotations are already in `runs/`. Running sections 5–8 spends money.

| Section | What it does | Costs money? |
|---|---|---|
| 1–2 | Setup, API keys | No |
| 3–4 | Build the batch files, estimate cost | No |
| 5 | **Smoke test**: 6 live requests per model, checked by the validator | A few cents |
| 6 | Submit the batches | Yes, the full run |
| 7 | Check status, download results (re-run until done) | No |
| 8 | Validate; resubmit faulty rows (repeat 7–8 until clean) | Only resubmitted rows |

## 1. Setup

In [ ]:
# !pip -q install -U openai anthropic tiktoken    # uncomment on a fresh machine or in Colab

import os, sys, json, glob, subprocess, time
from pathlib import Path

# Repository root: the first folder at or above the working directory that contains data/annotation_input.csv.
ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "annotation_input.csv").exists())
PKG   = ROOT / "llm_annotation"
INPUT = ROOT / "data" / "annotation_input.csv"
WORK  = PKG / "runs"                     # everything this notebook writes goes here
WORK.mkdir(parents=True, exist_ok=True)
assert (PKG / "cmv_llm_batch.py").exists(), f"cmv_llm_batch.py not found in {PKG}"
assert INPUT.exists(), f"Input CSV not found: {INPUT}"

sys.path.insert(0, str(PKG))
import importlib, cmv_llm_batch as L
importlib.reload(L)
TOOL = [sys.executable, str(PKG / "cmv_llm_batch.py")]

def run(args):
    r = subprocess.run(TOOL + [str(a) for a in args], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    r.check_returncode()
print("Setup OK")

## 2. API keys

Only needed for sections 5–8. Never paste keys into a cell. Set `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` as environment variables, or, in Colab, add them in the **Secrets** panel (key icon) with notebook access on.

In [ ]:
try:
    from google.colab import userdata          # Colab Secrets, if running in Colab
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        os.environ.setdefault(k, userdata.get(k) or "")
except ImportError:
    pass                                       # otherwise use environment variables
assert os.environ.get("OPENAI_API_KEY") and os.environ.get("ANTHROPIC_API_KEY"), "API keys not set"

from openai import OpenAI
import anthropic
oai = OpenAI()
ant = anthropic.Anthropic()
print("Clients ready")

## 3. Model settings and batch files

Lowest-reasoning settings, as requested:

- **GPT-6.1 Sol**: `reasoning_effort = "low"` (the lowest value this model accepts). It still reasons internally at `low`, and that hidden reasoning is billed as output, so `max_tokens` leaves room for it.
- **Claude Sonnet 5.5**: `effort = "low"` with `thinking = {"type": "between_tools"}`, which turns off up-front thinking. This model rejects `"disabled"`.

In [ ]:
MODELS = {
    "gpt":    dict(provider="openai",    model="gpt-6.1-sol",       effort="low", thinking=None,            max_tokens=4000),
    "claude": dict(provider="anthropic", model="claude-sonnet-5-5", effort="low", thinking="between_tools", max_tokens=1024),
}
SUBSET = "--main-only"     # 115 main-round OPs = 1,725 pairs

def folder(key, sub=""):
    p = WORK / key / sub
    p.mkdir(parents=True, exist_ok=True)
    return p

for key, m in MODELS.items():
    args = ["build", "--input", INPUT, "--provider", m["provider"], "--model", m["model"],
            "--reasoning-effort", m["effort"], "--max-tokens", m["max_tokens"],
            "--out-dir", folder(key, "batch_input"), SUBSET]
    if m["thinking"]:
        args += ["--thinking", m["thinking"]]
    print(f"===== {key}: {m['model']}")
    run(args)

def request_files(key):
    return sorted(glob.glob(str(folder(key, "batch_input") / "batch_main_*.jsonl")))

## 4. Cost estimate

Input tokens are counted exactly from the batch files (OpenAI's tokenizer; Claude's differs slightly). Output tokens are assumed, so the table shows a range. Both models cost **$1 input / $5 output per 1M tokens** at batch prices. Section 5 measures the real output tokens per request.

In [ ]:
open(WORK / "pricing.csv", "w").write("model,batch_input_per_1m,batch_output_per_1m\nbatch $1/$5,1,5\n")
for key in MODELS:
    for out_tokens in (100, 400, 1000):
        print(f"===== {key}: assuming {out_tokens} output tokens per request")
        run(["estimate", "--requests", *request_files(key), "--pricing", WORK / "pricing.csv",
             "--out-tokens", out_tokens])

## 5. Smoke test (strongly recommended)

Sends 6 real requests per model (2 per metric) **outside** the batch, at normal (non-batch) prices, a few cents in total. Each response goes through the same validator the batch results will use. If anything fails here, fix it before spending money on the full batch.

In [ ]:
def first_requests(key, n_per_metric=2):
    seen, picked = {}, []
    for f in request_files(key):
        for line in open(f, encoding="utf-8"):
            r = json.loads(line)
            metric = L.parse_custom_id(r["custom_id"])["metric"]
            if seen.get(metric, 0) < n_per_metric:
                seen[metric] = seen.get(metric, 0) + 1
                picked.append(r)
    return picked

def call_sync(provider, req):
    """Send one request synchronously and wrap the reply like a batch result line."""
    if provider == "openai":
        resp = oai.chat.completions.create(**req["body"])
        return {"custom_id": req["custom_id"],
                "response": {"status_code": 200, "body": resp.model_dump(mode="json")}, "error": None}
    params = dict(req["params"])
    try:
        msg = ant.messages.create(**params)
    except TypeError:   # older SDK without these fields: pass them through as raw JSON
        extra = {k: params.pop(k) for k in ("output_config", "thinking") if k in params}
        msg = ant.messages.create(**params, extra_body=extra)
    return {"custom_id": req["custom_id"],
            "result": {"type": "succeeded", "message": msg.model_dump(mode="json")}}

usage_seen = {}
for key, m in MODELS.items():
    print(f"\n===== {key}: {m['model']}")
    outs = []
    for req in first_requests(key):
        res = call_sync(m["provider"], req)
        metric = L.parse_custom_id(req["custom_id"])["metric"]
        out, usage, model, why = L.check_result(metric, res)
        status = "OK " if out else f"FAIL ({why})"
        print(f"{status} {req['custom_id']}")
        if out:
            print(f"     choice={out.get('choice')!r}  evidence={out.get('evidence_present', '-')}")
            print(f"     reasoning: {out['reasoning']}")
            print(f"     tokens in/out: {usage['prompt_tokens']} / {usage['completion_tokens']}")
            outs.append(usage["completion_tokens"])
    if outs:
        usage_seen[key] = sum(outs) / len(outs)
        print(f"Mean output tokens per request: {usage_seen[key]:.0f}")

print("\nRe-run section 4 with these output-token numbers for a measured cost estimate.")

In [ ]:
import itertools, pandas as pd, cmv_llm_batch as L
from pathlib import Path

csv  = INPUT

# 1. Main set exactly as the annotation notebook builds it
raw = pd.read_csv(csv, dtype={"post_id": str, "response_id": str}, encoding="utf-8-sig")
order = raw.drop_duplicates("post_id").set_index("post_id")["sample_post_order"].astype(int).to_dict()
master = [p for p, _ in sorted(order.items(), key=lambda kv: (kv[1], kv[0]))]
nb_main = {f"{p}|{lo}|{hi}"
           for p in master[5:]
           for lo, hi in itertools.combinations(sorted(raw[raw.post_id == p].response_id), 2)}

# 2. Main set as the LLM batch builder makes it
pairs, _ = L.build_pairs(L.load_sample(str(csv)), main_only=True)
llm_main = {x["pair_key"] for x in pairs}

print("Notebook main pairs:", len(nb_main), "| LLM pairs:", len(llm_main),
      "| identical:", nb_main == llm_main)

# 3. Against the pairs actually annotated
for f in sorted((ROOT / "data" / "human").glob("main_H*.csv")):
    keys = set(pd.read_csv(f, dtype=str)["pair_key"])
    print(f"{f.name}: {len(keys & llm_main)} main pairs annotated, "
          f"{len(keys - llm_main - (keys & nb_main))} outside the main set (should be 0)")

## 6. Submit the batches

**This spends money.** It submits every batch file for both models. The batch IDs are saved to `runs/state.json` on Drive, and re-running the cell never submits the same file twice.

In [ ]:
STATE_FILE = WORK / "state.json"

def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {"batches": []}

def save_state(s):
    STATE_FILE.write_text(json.dumps(s, indent=2))

def submit(key, path, round_no):
    """Submit one request file; record it in state.json. Skips files already submitted."""
    s = load_state()
    rel = str(Path(path).resolve().relative_to(ROOT))
    if any(b["file"] == rel for b in s["batches"]):
        print(f"already submitted: {Path(path).name}")
        return
    m = MODELS[key]
    if m["provider"] == "openai":
        f = oai.files.create(file=open(path, "rb"), purpose="batch")
        b = oai.batches.create(input_file_id=f.id, endpoint="/v1/chat/completions",
                               completion_window="24h", metadata={"run": f"{key}-r{round_no}"})
        bid = b.id
    else:
        reqs = [json.loads(line) for line in open(path, encoding="utf-8")]
        b = ant.messages.batches.create(requests=reqs)
        bid = b.id
    s["batches"].append({"key": key, "round": round_no, "file": rel, "batch_id": bid,
                         "submitted": time.strftime("%Y-%m-%d %H:%M"), "downloaded": False})
    save_state(s)
    print(f"submitted {key} round {round_no}: {Path(path).name} -> {bid}")

for key in MODELS:
    for f in request_files(key):
        submit(key, f, 0)

## 7. Check status and download results

Re-run this cell until every batch shows as downloaded. Batches can take up to 24 hours, though they often finish much sooner.

In [ ]:
# Helpers sections 7 and 8 need (copied from sections 3 and 6). No API calls.
MODELS = {
    "gpt":    dict(provider="openai",    model="gpt-6.1-sol",       effort="low", thinking=None,            max_tokens=4000),
    "claude": dict(provider="anthropic", model="claude-sonnet-5-5", effort="low", thinking="between_tools", max_tokens=1024),
}

def folder(key, sub=""):
    p = WORK / key / sub
    p.mkdir(parents=True, exist_ok=True)
    return p

def request_files(key):
    return sorted(glob.glob(str(folder(key, "batch_input") / "batch_main_*.jsonl")))

STATE_FILE = WORK / "state.json"

def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {"batches": []}

def save_state(s):
    STATE_FILE.write_text(json.dumps(s, indent=2))

def submit(key, path, round_no):
    """Submit one request file; record it in state.json. Skips files already submitted."""
    s = load_state()
    rel = str(Path(path).resolve().relative_to(ROOT))
    if any(b["file"] == rel for b in s["batches"]):
        print(f"already submitted: {Path(path).name}")
        return
    m = MODELS[key]
    if m["provider"] == "openai":
        f = oai.files.create(file=open(path, "rb"), purpose="batch")
        b = oai.batches.create(input_file_id=f.id, endpoint="/v1/chat/completions",
                               completion_window="24h", metadata={"run": f"{key}-r{round_no}"})
        bid = b.id
    else:
        reqs = [json.loads(line) for line in open(path, encoding="utf-8")]
        b = ant.messages.batches.create(requests=reqs)
        bid = b.id
    s["batches"].append({"key": key, "round": round_no, "file": rel, "batch_id": bid,
                         "submitted": time.strftime("%Y-%m-%d %H:%M"), "downloaded": False})
    save_state(s)
    print(f"submitted {key} round {round_no}: {Path(path).name} -> {bid}")

s = load_state()
print(f"{len(s['batches'])} batches on record:")
for e in s["batches"]:
    print(f"  {e['key']}  {e['batch_id']}  downloaded={e['downloaded']}")

In [ ]:
def download(entry):
    key, bid = entry["key"], entry["batch_id"]
    dest = folder(key, "results") / f"round{entry['round']:02d}_{bid}.jsonl"
    if MODELS[key]["provider"] == "openai":
        b = oai.batches.retrieve(bid)
        print(f"{key} {bid}: {b.status}  {b.request_counts}")
        if b.status not in ("completed", "failed", "expired", "cancelled"):
            return False
        with open(dest, "w", encoding="utf-8") as fh:
            for fid in (b.output_file_id, b.error_file_id):
                if fid:
                    fh.write(oai.files.content(fid).text)
    else:
        b = ant.messages.batches.retrieve(bid)
        print(f"{key} {bid}: {b.processing_status}  {b.request_counts}")
        if b.processing_status != "ended":
            return False
        with open(dest, "w", encoding="utf-8") as fh:
            for r in ant.messages.batches.results(bid):
                fh.write(json.dumps(r.model_dump(mode="json")) + "\n")
    print(f"   downloaded -> {dest.name}")
    return True

s = load_state()
for entry in s["batches"]:
    if not entry["downloaded"] and download(entry):
        entry["downloaded"] = True
save_state(s)
pending = [e for e in s["batches"] if not e["downloaded"]]
print(f"\n{len(pending)} batch(es) still running" if pending else "\nAll submitted batches downloaded.")

## 8. Validate, and resubmit faulty rows

Checks every response (valid JSON, an allowed label, consistent evidence answer, complete generation). Valid rows go to `runs/<model>/llm_output/llm_annotations_*.csv`. Faulty rows are written to a resubmission file, which this cell submits as a new batch.

Then go back to section 7, wait for it to finish, and run this cell again. Stop when both models report **All 10,350 requests valid**.

In [ ]:
s = load_state()
for key in MODELS:
    results = sorted(glob.glob(str(folder(key, "results") / "*.jsonl")))
    if not results:
        print(f"{key}: no results downloaded yet"); continue
    if any(not e["downloaded"] for e in s["batches"] if e["key"] == key):
        print(f"{key}: a batch is still running; finish section 7 first"); continue
    next_round = max(e["round"] for e in s["batches"] if e["key"] == key) + 1
    print(f"===== {key}")
    run(["validate", "--requests", *request_files(key), "--results", *results,
         "--out-dir", folder(key, "llm_output"), "--round", next_round])
    resub = folder(key, "llm_output") / f"resubmit_round{next_round:02d}.jsonl"
    if resub.exists() and resub.stat().st_size > 0:
        submit(key, resub, next_round)

## Done

The final annotation files are in `llm_annotation/runs/gpt/llm_output/` and `llm_annotation/runs/claude/llm_output/`; `analysis/02_studies.ipynb` reads them. `runs/state.json` records every batch ID, model and submission time.